# 🧠 Home NIDS — Train Model (ตัวเทรนหลัก)

Notebook นี้คือที่เทรนโมเดลหลักของโปรเจกต์ รันจากบนลงล่างได้เลย (**Kernel → Restart & Run All**)

| ขั้น | ทำอะไร | ได้อะไรออกมา |
|---|---|---|
| 0 | ตั้งค่า (ดึงจาก `config/config.yaml` + import โค้ดของ microservice) | ค่าตั้ง, `RUN_DIR` |
| 1 | แปลง `.pcap` → CSV (`training/pcap/normal/`, `training/pcap/test/`) | `normal_csvs`, `test_csvs` |
| 2 | โหลดข้อมูลเทรน (traffic ปกติ) แล้วคำนวณ 36 feature | `X` (n × 36) |
| 3 | ดูการกระจายของ feature | รูป 2 รูป |
| 4 | เทรน StandardScaler + Isolation Forest | `scaler`, `model` |
| 5 | สร้างชุดทดสอบที่มี label (ปกติ / โจมตี) | `X_test`, `y_test`, `s_test` |
| 6 | เลือก threshold | `THRESHOLD` |
| 7 | วัดผล: **Confusion Matrix (TP/FP/TN/FN), Accuracy, Precision, Recall, F1, FPR, ROC-AUC** + เกณฑ์ผ่าน README | `m`, `PASS` |
| 8 | บันทึกโมเดลลง `system/models/` → detection-service ใช้ต่อเมื่อรีสตาร์ท | 3 ไฟล์ในโมเดล |

> **วิธีเพิ่ม dataset:** วาง `.pcap` traffic ปกติใน `training/pcap/normal/`, ไฟล์ที่มีการโจมตีใน `training/pcap/test/`
> แล้วเขียนกฎ label (IP ผู้โจมตี + ช่วงเวลา) ใน `training/pcap/labels.csv`
> ถ้ายังไม่มีไฟล์ notebook ยังรันได้ด้วยข้อมูลจำลอง

## 0. ตั้งค่า
ค่าหลักมาจาก `config/config.yaml` (ไฟล์เดียวกับที่ microservice ใช้) ส่วนสวิตช์ของการเทรนแก้ได้ในเซลล์นี้

In [ ]:
# * ===================== ขั้น 0: ตั้งค่า =====================
# * เซลล์นี้ยังไม่ได้เทรนอะไร แค่เตรียมของทุกอย่างไว้ให้เซลล์ถัดๆ ไปใช้

# โหลดเครื่องมือพื้นฐานของ Python: json ไว้เขียนไฟล์ผล, shutil ไว้ก๊อปไฟล์, sys ไว้บอกว่าจะหาโค้ดที่ไหน, time ไว้จับเวลา
import json, shutil, sys, time
# Counter เอาไว้นับ เช่น นับว่ามี TCP กี่ตัว UDP กี่ตัว
from collections import Counter
# Path เอาไว้จัดการชื่อไฟล์และโฟลเดอร์ ใช้ได้ทั้ง Windows และ Linux
from pathlib import Path

# * ---- 0.1 หาว่าโฟลเดอร์โปรเจกต์อยู่ตรงไหนในเครื่อง ----
# ที่อยู่สำรอง ไว้ใช้ถ้าหาจากโฟลเดอร์ที่เปิดอยู่ไม่เจอ
_FALLBACK_ROOT = Path(r"C:\Project_AI\Project-AI-IDS")
# ลองไล่ดูทีละที่: โฟลเดอร์ที่เปิดอยู่ตอนนี้ -> โฟลเดอร์ที่อยู่เหนือขึ้นไป (กรณีเปิดจากใน training/) -> ที่อยู่สำรอง
for _cand in (Path.cwd(), Path.cwd().parent, _FALLBACK_ROOT):
    # ถ้าโฟลเดอร์ไหนมีไฟล์ system/shared/features.py แปลว่าใช่โฟลเดอร์โปรเจกต์แน่นอน
    if (_cand / "system" / "shared" / "features.py").exists():
        # จำโฟลเดอร์นี้ไว้ในชื่อ ROOT
        ROOT = _cand
        # เจอแล้วก็เลิกหา
        break
# ถ้าไล่ครบทุกที่แล้วยังไม่เจอ ให้หยุดทำงานเลย แล้วบอกว่าตอนนี้เปิดอยู่ที่โฟลเดอร์ไหน
else:
    # หยุดโปรแกรมพร้อมข้อความ error
    raise RuntimeError(f"หา root ของโปรเจกต์ไม่เจอ -- cwd ตอนนี้คือ {Path.cwd()}")
# บอก Python ว่าให้หาโค้ดในโฟลเดอร์โปรเจกต์ด้วย จะได้ import ของใน training/ ได้
sys.path.insert(0, str(ROOT))
# บอก Python ให้หาในโฟลเดอร์ system ด้วย จะได้ import shared/ ซึ่งเป็นโค้ดชุดเดียวกับที่ระบบจริงใช้
sys.path.insert(0, str(ROOT / "system"))

# numpy = ตัวคำนวณตัวเลขเป็นตารางใหญ่ๆ ใช้ทั้ง notebook
import numpy as np
# matplotlib = ตัววาดกราฟ
import matplotlib.pyplot as plt
# Markdown, display = ไว้โชว์ตารางสวยๆ ใน notebook
from IPython.display import Markdown, display
# font_manager = ไว้ดูว่าเครื่องนี้มีฟอนต์อะไรบ้าง
from matplotlib import font_manager

# เก็บรายชื่อฟอนต์ทั้งหมดที่เครื่องนี้มี
_fonts = {f.name for f in font_manager.fontManager.ttflist}
# เลือกฟอนต์ไทยตัวแรกที่เครื่องมีมาใช้ในกราฟ ถ้าไม่มีสักตัวใช้ DejaVu Sans แทน (ภาษาไทยจะกลายเป็นกล่องสี่เหลี่ยม)
plt.rcParams["font.family"] = [f for f in ("Tahoma", "Leelawadee UI", "Noto Sans Thai", "Loma") if f in _fonts] or ["DejaVu Sans"]

# * ---- 0.2 ดึงโค้ดที่ต้องใช้มาจากไฟล์อื่น ----
# จาก system/shared/config.py: load_config = อ่านไฟล์ตั้งค่า, resolve_path = แปลงที่อยู่ไฟล์ให้นับจากโฟลเดอร์โปรเจกต์
from shared.config import load_config, resolve_path
# จาก system/shared/features.py: รายชื่อ 36 feature เรียงตามลำดับ กับเลขเวอร์ชันของสูตร (ตอนนี้ 6)
from shared.features import FEATURE_NAMES, FEATURE_VERSION
# จาก training/pipeline/generate_data.py: สร้าง traffic ปลอม / อ่าน CSV / เขียน CSV
from training.pipeline.generate_data import generate, load_packets_csv, save_packets_csv
# จาก training/pipeline/pcap_to_csv.py: แปลงไฟล์ Wireshark ทั้งโฟลเดอร์เป็น CSV / ดูรายชื่อไฟล์ pcap
from training.pipeline.pcap_to_csv import convert_folder, list_pcaps
# จาก training/pipeline/common.py: หัวใจของการเทรน (คิด feature, เทรน, ให้คะแนน, หาเส้นตัด, บันทึกโมเดล)
from training.pipeline.common import (build_features, train_model, anomaly_scores,
                                      recommend_threshold, save_artifacts)
# จาก training/pipeline/scenarios.py: การโจมตีปลอมสำหรับทดสอบ (ping 4 แบบ, nmap 6 แบบ)
from training.pipeline.scenarios import (make_scenarios, is_ping_scenario, ping_request_scores,
                                         score_scenarios, scan_report, SCAN_DETECT_WITHIN)
# จาก training/pipeline/evaluate.py: ตัววัดผล ติดเฉลย ทำตาราง วาดกราฟ (เรียกสั้นๆ ว่า ev)
from training.pipeline import evaluate as ev

# * ---- 0.3 อ่านค่าจากไฟล์ตั้งค่า config/config.yaml (ไฟล์เดียวกับที่ระบบจริงใช้) ----
# อ่านไฟล์ตั้งค่าทั้งไฟล์มาเก็บไว้
cfg = load_config()
# หยิบเฉพาะส่วน training มาเก็บไว้ในชื่อสั้นๆ ว่า T
T = cfg["training"]

# ดูย้อนหลังกี่วินาทีตอนคิด feature แบบบริบท (5 วินาที) ต้องเท่ากับที่ระบบจริงใช้
WINDOW_SEC   = cfg["feature"]["window_sec"]
# จะสร้างต้นไม้ตัดสินกี่ต้น (1000 ต้น)
N_ESTIMATORS = T["n_estimators"]
# เลขตั้งต้นของการสุ่ม (42) ใส่เลขเดิม สุ่มออกมาเหมือนเดิมทุกครั้ง
SEED         = T["seed"]
# เส้นตัดต้องสูงกว่าคะแนนของ traffic ปกติกี่เปอร์เซ็นต์ (99.8%)
FP_QUANTILE  = T["fp_quantile"]
# อ่าน pcap ไม่เกินกี่ packet ต่อไฟล์ กันเครื่องแรมเต็ม
MAX_PACKETS  = T["max_packets_per_pcap"]
# โฟลเดอร์ใส่ไฟล์ pcap ที่เป็น traffic ปกติ (ใช้เทรน)
NORMAL_DIR   = resolve_path(T["pcap_normal_dir"])
# โฟลเดอร์ใส่ไฟล์ pcap ที่มีการโจมตีปน (ใช้สอบ)
TEST_DIR     = resolve_path(T["pcap_test_dir"])
# ไฟล์ที่เขียนบอกว่า IP ไหน ช่วงเวลาไหน คือการโจมตี (ใช้ทำเฉลย)
LABELS_FILE  = resolve_path(T["labels_file"])
# โฟลเดอร์เก็บ CSV ที่แปลงจาก pcap แล้ว รอบหน้าจะได้ไม่ต้องแปลงใหม่
PCAP_CSV_DIR = resolve_path(T["pcap_csv_dir"])
# บางไฟล์ให้อ่านเฉพาะช่วงเวลา เช่น ไฟล์วันศุกร์อ่านแค่ช่วงบ่ายที่มีการสแกน ถ้าไม่ได้ตั้งไว้ใช้ค่าว่าง
TIME_WINDOWS = T.get("pcap_time_windows") or {}

# * ---- 0.4 สวิตช์ที่แก้เองได้ ----
# ! ไฟล์ CSV ที่ใส่เพิ่มต้องอัดด้วยโปรแกรม capture รุ่น v5 ขึ้นไป (บรรทัดแรกต้องมีคำว่า tcp_window)
# ! ไฟล์บ้านชุดเก่า (home_discord_normal, soak_*) ไม่มีช่องข้อมูลใหม่ ห้ามใส่
# ไฟล์ traffic บ้านแบบเต็ม (ถ้าเครื่องไหนมีก็ใช้ ไม่มีก็ข้าม)
CAPTURED_CSV = resolve_path("data/captured_full_v5.csv")
# ไฟล์ traffic ปกติเพิ่มเติม (ต้องไม่มี ping หรือการสแกนปน)
EXTRA_NORMAL_CSVS = [
    resolve_path("data/captured_v5_active1_train.csv"),            # บ้าน: ตอนใช้คอม 17 พ.ย. 22:30-23:37
    resolve_path("data/captured_v5_night1_train.csv"),             # บ้าน: เปิดทิ้งไว้ 18 พ.ย. 05:22-17:54
    resolve_path("data/pcap_csv_v5/normal__Monday-WorkingHours.csv"),   # dataset CIC วันจันทร์ (ปกติล้วน)
]
# ไฟล์ที่ใช้สอบโมเดลเพิ่มเติม (จะติดเฉลยด้วย labels.csv)
EXTRA_TEST_CSVS = [
    resolve_path("data/captured_v5_active1_test.csv"),             # บ้าน: ช่วงท้าย 23:37-00:06 (ไม่ทับกับชุดเทรน)
    resolve_path("data/captured_v5_night1_test.csv"),              # บ้าน: 18 พ.ย. 17:54-23:16 ไว้นับแจ้งเตือนหลอกต่อชั่วโมง
    resolve_path("data/pcap_csv_v5/test__Friday-WorkingHours.csv"),     # dataset CIC วันศุกร์ (มีการสแกน port จริง)
]

# ใช้ traffic ปลอมที่สร้างเองด้วยไหม (ควรเปิด เพราะมี ICMP ที่ข้อมูลจริงแทบไม่มี)
USE_SYNTHETIC        = True
# สร้าง traffic ปลอมเพิ่มอีกกี่ packet (0 = ไม่เพิ่ม)
EXTRA_SYNTHETIC      = 40_000
# ใช้ไฟล์ traffic บ้านกับไฟล์ปกติเพิ่มเติมด้วยไหม
USE_CAPTURED         = True
# แต่ละไฟล์เก็บได้ไม่เกินกี่แถวต่อหนึ่งโปรโตคอล กัน UDP ของเกม/Discord มีเยอะจนกลบอย่างอื่น (None = เก็บหมด)
MAX_TRAIN_PER_SOURCE = 200_000
# ICMP ที่ไม่ใช่ ping ต้องมีอย่างน้อย 1% ของข้อมูลเทรน ไม่งั้นโมเดลไม่สนใจ ICMP แล้ว ping จะไม่ดูแปลก
ICMP_MIN_FRAC        = 0.01
# ตัด ping ออกจากข้อมูลเทรน ไม่งั้นโมเดลจะจำว่า ping เป็นเรื่องปกติ
DROP_ICMP_ECHO       = True
# ใส่ ping กับ nmap ปลอมเข้าไปในข้อสอบด้วย
USE_SIM_TEST         = True
# True = แปลง pcap ใหม่ทุกครั้ง แม้เคยแปลงไว้แล้ว (ช้า)
FORCE_RECONVERT      = False
# วิธีเลือกเส้นตัด: "recommended" (ใช้จริง), "best_f1" (แอบดูข้อสอบ ตัวเลขสวยเกินจริง) หรือใส่ตัวเลขเอง
THRESHOLD_MODE       = "recommended"
# True = ถ้าโมเดลสอบไม่ผ่านเกณฑ์ README จะไม่ยอมบันทึก
REQUIRE_PASS         = False

# * ---- 0.5 เตรียมโฟลเดอร์เก็บผลของรอบนี้ ----
# ตั้งชื่อโฟลเดอร์ผลตามวันเวลาที่รัน เช่น results/train_20261118_234059
RUN_DIR = resolve_path("results") / time.strftime("train_%Y%m%d_%H%M%S")
# สร้างโฟลเดอร์นั้นขึ้นมา (มีอยู่แล้วก็ไม่เป็นไร)
RUN_DIR.mkdir(parents=True, exist_ok=True)
# เริ่มเนื้อหารายงาน report.md ด้วยหัวเรื่องบอกวันเวลา
_REPORT = [f"# ผลการเทรน {time.strftime('%Y-%m-%d %H:%M:%S')}\n"]

# ฟังก์ชันเพิ่มหัวข้อลงรายงาน
def report(title, text=""):
    """เพิ่มหัวข้อลง report.md (เขียนทั้งไฟล์ทุกครั้ง ถ้ารันพังกลางทางยังได้ผลถึงขั้นล่าสุด)"""
    # ต่อหัวข้อกับเนื้อหาใหม่เข้าไปท้ายรายงาน
    _REPORT.append(f"## {title}\n\n{text}\n")
    # เขียนรายงานทั้งหมดลงไฟล์ใหม่ทุกครั้ง ถ้ารันพังกลางทาง ไฟล์ก็ยังมีผลถึงขั้นล่าสุด
    (RUN_DIR / "report.md").write_text("\n".join(_REPORT), encoding="utf-8")

# ฟังก์ชันบันทึกรูปกราฟลงโฟลเดอร์ผล
def save_fig(fig, name):
    """บันทึกรูปลง RUN_DIR (ต้องเรียกก่อน plt.show() เพราะ show แล้วรูปถูกล้าง)"""
    # เซฟเป็นไฟล์รูป ต้องเรียกก่อนโชว์กราฟ เพราะโชว์แล้วรูปจะถูกล้าง
    fig.savefig(RUN_DIR / name, dpi=120, bbox_inches="tight")

# ฟังก์ชันเช็กว่าไฟล์ CSV เป็นรุ่นใหม่ (v5) ไหม
def has_v5_fields(path):
    """CSV นี้มีคอลัมน์ของ capture v5 ไหม (อ่านแค่บรรทัดแรก ไม่โหลดทั้งไฟล์)"""
    # เปิดไฟล์อ่าน (ตัวอักษรเพี้ยนก็ข้ามไป ไม่ให้พัง)
    with open(path, encoding="utf-8", errors="ignore") as f:
        # อ่านแค่บรรทัดแรกที่เป็นชื่อคอลัมน์ ถ้ามี tcp_window = รุ่นใหม่
        return "tcp_window" in f.readline()

# จดค่าตั้งทั้งหมดของรอบนี้ลงรายงาน ไว้ย้อนเทียบกับรอบอื่นได้
report("ตั้งค่า", ev.md_table([{
    "window_sec": WINDOW_SEC, "n_estimators": N_ESTIMATORS, "seed": SEED, "fp_quantile": FP_QUANTILE,
    "max_packets/pcap": MAX_PACKETS, "USE_SYNTHETIC": USE_SYNTHETIC, "USE_CAPTURED": USE_CAPTURED,
    "USE_SIM_TEST": USE_SIM_TEST, "DROP_ICMP_ECHO": DROP_ICMP_ECHO, "THRESHOLD_MODE": THRESHOLD_MODE,
    "MAX_TRAIN_PER_SOURCE": MAX_TRAIN_PER_SOURCE, "ICMP_MIN_FRAC": ICMP_MIN_FRAC,
    "EXTRA_SYNTHETIC": EXTRA_SYNTHETIC, "REQUIRE_PASS": REQUIRE_PASS, "FEATURE_VERSION": FEATURE_VERSION}])
    + f"\n\nextra normal: `{[p.name for p in EXTRA_NORMAL_CSVS]}` | extra test: `{[p.name for p in EXTRA_TEST_CSVS]}`"
    + f"\n\ntime windows: `{TIME_WINDOWS}`")
# โชว์ว่าผลรอบนี้เก็บไว้ที่ไหน
print("เก็บผลรอบนี้ที่:", RUN_DIR)
# โชว์ว่าใช้ไฟล์ตั้งค่าตัวไหน
print("config:", resolve_path("config/config.yaml"))
# โชว์ค่าหลัก ควรเห็น features=36 (v6)
print(f"window={WINDOW_SEC}s  n_estimators={N_ESTIMATORS}  features={len(FEATURE_NAMES)} (v{FEATURE_VERSION})")

## 1. แปลง pcap → CSV
อ่านไฟล์ใน `training/pcap/normal/` และ `training/pcap/test/` แปลงด้วยฟังก์ชันเดียวกับ capture-service (`packet_to_meta`) เก็บไว้ที่ `data/pcap_csv_v5/` (ครั้งถัดไปใช้ของเดิม ไม่ต้องแปลงซ้ำ)

In [ ]:
# * ===================== ขั้น 1: แปลงไฟล์ Wireshark (.pcap) เป็น CSV =====================
# * ได้ CSV ที่ 1 แถว = 1 packet ใช้วิธีแปลงเดียวกับที่ระบบจริงใช้ตอนดัก packet

# โชว์รายชื่อไฟล์ pcap ที่เป็น traffic ปกติ ถ้าไม่มีเลยเขียนว่า "ไม่มี"
print("pcap traffic ปกติ :", [p.name for p in list_pcaps(NORMAL_DIR)] or "ไม่มี")
# โชว์รายชื่อไฟล์ pcap ที่ใช้สอบ
print("pcap สำหรับทดสอบ :", [p.name for p in list_pcaps(TEST_DIR)] or "ไม่มี")
# จดเวลาเริ่ม ไว้ดูว่าแปลงนานแค่ไหน
t0 = time.time()
# แปลง pcap ปกติทุกไฟล์เป็น CSV (เคยแปลงแล้วก็ใช้ของเดิม) ได้รายชื่อไฟล์ CSV กลับมา
normal_csvs = convert_folder(NORMAL_DIR, PCAP_CSV_DIR, MAX_PACKETS, FORCE_RECONVERT, windows=TIME_WINDOWS)
# แปลง pcap ข้อสอบทุกไฟล์เป็น CSV เหมือนกัน
test_csvs   = convert_folder(TEST_DIR,   PCAP_CSV_DIR, MAX_PACKETS, FORCE_RECONVERT, windows=TIME_WINDOWS)
# เลือกเฉพาะไฟล์สอบเพิ่มเติมที่มีอยู่จริงในเครื่อง
extra_test  = [p for p in EXTRA_TEST_CSVS if p.exists()]
# รวมสองกลุ่มเข้าด้วยกัน แล้วตัดชื่อไฟล์ที่ซ้ำออก (กันไฟล์เดียวถูกนับสองรอบ)
test_csvs   = list(dict.fromkeys(test_csvs + extra_test))
# โชว์ว่ามีไฟล์สอบเพิ่มเติมตัวไหนบ้าง
print("CSV ทดสอบเพิ่มเติม:", [p.name for p in extra_test] or "ไม่มี")
# โชว์เวลาที่ใช้แปลง
print(f"เสร็จใน {time.time() - t0:.1f}s")

## 2. โหลดข้อมูลเทรน (traffic ปกติเท่านั้น)
แต่ละไฟล์คำนวณ feature แยกกัน (sliding window ไม่ข้ามไฟล์) แล้วรวมเป็นชุดเดียว

In [ ]:
# * ===================== ขั้น 2: โหลดข้อมูลเทรน (traffic ปกติเท่านั้น) =====================
# * ทีละไฟล์: อ่าน -> ตัด ping -> เรียงตามเวลา -> แปลงเป็น 36 ตัวเลข -> ลดจำนวนตามโปรโตคอล
# * จากนั้นต่อทุกไฟล์เป็นตารางเดียวชื่อ X แล้วเติม ICMP ให้พอ

# * ---- 2.1 รวมรายชื่อไฟล์ที่จะใช้เทรน ----
# เริ่มจากรายชื่อว่างๆ
sources = []
# ถ้าเปิดใช้ traffic ปลอม
if USE_SYNTHETIC:
    # ไฟล์ traffic ปลอมอยู่ที่ data/normal_packets.csv
    syn = resolve_path(T["packets_csv"])
    # ถ้ายังไม่มีไฟล์ หรือเป็นไฟล์รุ่นเก่า
    if not syn.exists() or not has_v5_fields(syn):
        # สร้าง traffic ปลอม 8,000 packet ใหม่แล้วเขียนลงไฟล์
        save_packets_csv(generate(T["n_packets"], SEED), syn)
        # บอกว่าสร้างไฟล์ใหม่แล้ว
        print(f"สร้าง synthetic ใหม่ด้วย generator v5: {syn}")
    # ใส่ไฟล์ปลอมเป็นแหล่งข้อมูลตัวแรก
    sources.append(syn)
# ต่อด้วย CSV ที่แปลงจาก pcap ปกติในขั้น 1
sources += normal_csvs
# ถ้าเปิดใช้ข้อมูลบ้านและไฟล์เพิ่มเติม
if USE_CAPTURED:
    # ไล่ดูทีละไฟล์: ไฟล์บ้านแบบเต็ม แล้วตามด้วยไฟล์เพิ่มเติม
    for extra_csv in [CAPTURED_CSV] + EXTRA_NORMAL_CSVS:
        # ถ้าไฟล์มีอยู่จริง
        if extra_csv.exists():
            # ใส่เข้ารายชื่อ
            sources.append(extra_csv)
        # ถ้าไม่มีไฟล์
        else:
            # บอกว่าหาไม่เจอแล้วข้ามไป
            print(f"⚠️ ไม่พบ {extra_csv.name} (ข้าม)")
# ตัดชื่อไฟล์ที่ซ้ำออก (กันไฟล์เดียวถูกใช้สองรอบ)
sources = list(dict.fromkeys(sources))
# ถ้าไม่มีไฟล์ให้เทรนเลยสักไฟล์ หยุดเลยพร้อมบอกวิธีแก้
assert sources, "ไม่มีข้อมูลเทรน: วาง pcap ใน training/pcap/normal/ หรือตั้ง USE_SYNTHETIC = True"

# * ---- 2.2 จัดการทีละไฟล์ ----
# สร้างตัวสุ่มที่ตั้งต้นด้วยเลขเดิม รันกี่ครั้งก็สุ่มได้แถวเดิม
rng = np.random.default_rng(SEED)
# blocks = ตารางตัวเลขของแต่ละไฟล์, summary = แถวสรุปของแต่ละไฟล์
blocks, summary = [], []
# วนทีละไฟล์
for src in sources:
    # ถ้าเป็น CSV รุ่นเก่า ให้เตือนว่าช่องข้อมูลใหม่จะเป็น 0 หมด
    if str(src).endswith(".csv") and not has_v5_fields(src):
        # พิมพ์คำเตือน (ยังทำต่อได้ แต่ข้อมูลไฟล์นี้จะไม่ครบ)
        print(f"⚠️ {Path(src).name} ไม่มี field v5 (อัด/แปลงด้วยโค้ดรุ่นเก่า) -> feature v5 จะเป็น 0 ทั้งหมด ควรแปลง/อัดใหม่")
    # อ่านไฟล์ ได้รายการ packet (1 packet = ข้อมูล 1 ชุด เช่น IP, port, ขนาด, เวลา)
    pk = load_packets_csv(src)
    # จำไว้ว่าทั้งไฟล์มีกี่ packet
    n0 = len(pk)
    # ถ้าเปิดสวิตช์ตัด ping
    if DROP_ICMP_ECHO:
        # เก็บไว้เฉพาะ packet ที่ไม่ใช่ ping (ping = ICMP ชนิด 8 ขาไป และชนิด 0 ขากลับ)
        pk = [p for p in pk if not (p["proto"] == "ICMP" and p["icmp_type"] in (0, 8))]
    # นับว่าตัด ping ออกไปกี่ตัว
    n_ping = n0 - len(pk)
    # เรียง packet ตามเวลา เพราะตัวคิด feature ต้องดูเหตุการณ์ไล่ตามลำดับเวลา
    pk.sort(key=lambda p: p["ts"])
    # แปลงทุก packet เป็นตัวเลข 36 ตัว ด้วยสูตรเดียวกับระบบจริง (คิดจากไฟล์เต็มก่อน ค่าบริบทจะได้ไม่เพี้ยน)
    Xb = build_features(pk, WINDOW_SEC)

    # ถ้าตั้งเพดานจำนวนแถวไว้ ค่อยมาลดจำนวนตรงนี้ (หลังคิด feature เสร็จแล้ว)
    if MAX_TRAIN_PER_SOURCE:
        # ดึงชื่อโปรโตคอลของทุก packet มาเรียงเป็นแถว
        protos = np.array([p["proto"] for p in pk])
        # รายการเลขแถวที่จะเก็บไว้
        keep = []
        # วนทีละโปรโตคอลที่มีในไฟล์ (TCP, UDP, ICMP, อื่นๆ)
        for pr in np.unique(protos):
            # หาว่าแถวไหนบ้างเป็นโปรโตคอลนี้
            idx_p = np.flatnonzero(protos == pr)
            # ถ้ามีเกินเพดาน
            if len(idx_p) > MAX_TRAIN_PER_SOURCE:
                # สุ่มหยิบมาแค่เท่าเพดาน (ไม่หยิบซ้ำ)
                idx_p = rng.choice(idx_p, MAX_TRAIN_PER_SOURCE, replace=False)
            # เก็บเลขแถวของโปรโตคอลนี้ไว้
            keep.append(idx_p)
        # รวมเลขแถวทุกโปรโตคอลแล้วเรียงกลับตามเวลา (ไฟล์ว่างได้รายการว่าง)
        idx = np.sort(np.concatenate(keep)) if keep else np.arange(0)
        # ถ้ามีการตัดแถวออกจริง
        if len(idx) < len(Xb):
            # เก็บเฉพาะแถวที่เลือกในตารางตัวเลข
            Xb = Xb[idx]
            # ตัดรายการ packet ให้ตรงกัน จะได้นับในตารางสรุปถูก
            pk = [pk[i] for i in idx]

    # เก็บตารางตัวเลขของไฟล์นี้ไว้
    blocks.append(Xb)
    # นับว่าเหลือ packet แต่ละโปรโตคอลกี่ตัว
    c = Counter(p["proto"] for p in pk)
    # จดแถวสรุปของไฟล์นี้: ทั้งไฟล์ / ตัด ping / ใช้เทรน / แยกโปรโตคอล
    summary.append({"ไฟล์": Path(src).name, "packets ทั้งไฟล์": n0, "ตัด ping": n_ping,
                    "ใช้เทรน": len(Xb), "TCP": c["TCP"], "UDP": c["UDP"], "ICMP": c["ICMP"], "อื่นๆ": c["OTHER"]})

# * ---- 2.3 traffic ปลอมเพิ่มเติม (ตัวสร้างไม่สร้าง ping อยู่แล้ว ไม่ต้องตัด) ----
# ถ้าตั้งให้สร้างเพิ่ม
if EXTRA_SYNTHETIC:
    # สร้าง traffic ปลอมอีก 40,000 packet ด้วยเลขสุ่มคนละชุดกับไฟล์แรก
    extra_syn = generate(EXTRA_SYNTHETIC, SEED + 735)
    # แปลงเป็น 36 ตัวเลขแล้วเก็บไว้
    blocks.append(build_features(extra_syn, WINDOW_SEC))
    # นับจำนวนแต่ละโปรโตคอล
    c = Counter(p["proto"] for p in extra_syn)
    # จดแถวสรุปของชุดนี้
    summary.append({"ไฟล์": f"synthetic เพิ่ม (seed {SEED + 735})", "packets ทั้งไฟล์": len(extra_syn), "ตัด ping": 0,
                    "ใช้เทรน": len(extra_syn), "TCP": c["TCP"], "UDP": c["UDP"], "ICMP": c["ICMP"], "อื่นๆ": c["OTHER"]})
# เอาตารางของทุกไฟล์มาต่อกันเป็นตารางใหญ่ตารางเดียว ชื่อ X (แถว = packet, คอลัมน์ = 36 feature)
X = np.vstack(blocks)

# * ---- 2.4 เติม ICMP ให้ถึง 1% โดยก๊อปแถว ICMP ที่มีอยู่ ----
# สูตร: อยากได้ (มีอยู่ + เติม) / (ทั้งหมด + เติม) = 1%  ->  เติม = (1% x ทั้งหมด - มีอยู่) / (1 - 1%)
# หาว่าคอลัมน์ "เป็น ICMP ไหม" อยู่คอลัมน์ที่เท่าไร
icmp_col = FEATURE_NAMES.index("proto_icmp")
# หาว่าแถวไหนบ้างเป็น ICMP (คอลัมน์นี้มีค่า 1)
icmp_rows = np.flatnonzero(X[:, icmp_col] > 0.5)
# จำว่ามี ICMP อยู่เดิมกี่แถว และตั้งตัวนับแถวที่เติมเป็น 0
n_icmp0, n_added = len(icmp_rows), 0
# ถ้ามี ICMP อยู่บ้าง แต่ยังไม่ถึง 1%
if ICMP_MIN_FRAC and n_icmp0 and n_icmp0 / len(X) < ICMP_MIN_FRAC:
    # คิดว่าต้องเติมกี่แถวตามสูตรข้างบน (ปัดขึ้น)
    n_added = int(np.ceil((ICMP_MIN_FRAC * len(X) - n_icmp0) / (1 - ICMP_MIN_FRAC)))
    # สุ่มหยิบแถว ICMP ที่มีอยู่มาก๊อปต่อท้าย X (หยิบซ้ำได้)
    X = np.vstack([X, X[rng.choice(icmp_rows, n_added, replace=True)]])
# ถ้าไม่มี ICMP เลยสักแถว ก๊อปไม่ได้
elif ICMP_MIN_FRAC and not n_icmp0:
    # เตือนว่า ping จะจับไม่ได้ ให้เปิดใช้ traffic ปลอม
    print("⚠️ ไม่มี ICMP ในข้อมูลเทรนเลย -> ping จะไม่ถูกแยก (เปิด USE_SYNTHETIC = True)")
# เขียนข้อความสรุปเรื่อง ICMP เช่น "159 แถว -> ทำสำเนาเพิ่ม 7,353 แถว"
icmp_note = (f"ICMP (ไม่ใช่ ping): {n_icmp0:,} แถว ({n_icmp0 / max(len(X) - n_added, 1):.4%})"
             + (f" -> ทำสำเนาเพิ่ม {n_added:,} แถว ให้ได้ {ICMP_MIN_FRAC:.2%}" if n_added else ""))

# โชว์ตารางสรุปทุกไฟล์
display(Markdown(ev.md_table(summary)))
# โชว์ข้อความเรื่อง ICMP
print(icmp_note)
# โชว์ขนาดตาราง X สุดท้าย
print("รวม", X.shape[0], "packets x", X.shape[1], "features")
# จดทั้งหมดนี้ลงรายงาน
report("ข้อมูลเทรน (traffic ปกติ)", ev.md_table(summary) + f"\n\n{icmp_note}\n\nรวม {X.shape[0]:,} packets x {X.shape[1]} features")

## 3. ดูการกระจายของ feature
เช็กด้วยตาว่าข้อมูลสมเหตุสมผล เช่นไม่มีคอลัมน์ที่เป็น 0 ทั้งหมด (อาการของไฟล์ที่ไม่มี field v5) เซลล์นี้ไม่มีผลต่อโมเดล

In [ ]:
# * ===================== ขั้น 3: วาดกราฟดูหน้าตาข้อมูล =====================
# * เซลล์นี้แค่ให้ดูด้วยตา ไม่ได้เปลี่ยนข้อมูล ไม่มีผลกับโมเดล

# เตรียมพื้นที่วาดกราฟ 8 ช่อง (2 แถว x 4 คอลัมน์)
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
# จับคู่ช่องกราฟแต่ละช่องกับ feature ที่เป็นตัวเลขต่อเนื่อง 8 ตัว
for ax, name in zip(axes.ravel(), ["length", "ttl", "ctx_pkts", "ctx_dst_ports",
                                   "ctx_dst_ips", "ctx_low_ports", "ctx_unans_ports", "ctx_unans_ports_long"]):
    # วาดกราฟแท่งนับความถี่ (แบ่งเป็น 40 ช่วง) ของคอลัมน์นั้น
    ax.hist(X[:, FEATURE_NAMES.index(name)], bins=40, color="tab:blue")
    # ใส่ชื่อกราฟ และให้แกนตั้งเป็นสเกล log (ค่าส่วนใหญ่กองอยู่ที่ต่ำ ถ้าไม่ใช้ log จะมองไม่เห็นแท่งเล็ก)
    ax.set_title(name); ax.set_yscale("log")
# จัดระยะไม่ให้กราฟทับกัน -> เซฟรูป -> โชว์
fig.tight_layout(); save_fig(fig, "features_hist.png"); plt.show()

# เลือก feature ที่เป็นแบบ 0/1 (โปรโตคอล, TCP flag, กลุ่ม port)
binary = [n for n in FEATURE_NAMES if n.startswith(("proto_", "flag_", "port_"))]
# เตรียมพื้นที่วาดกราฟอีกรูป
fig2 = plt.figure(figsize=(14, 3))
# วาดกราฟแท่ง: ค่าเฉลี่ยของคอลัมน์ 0/1 = มีกี่เปอร์เซ็นต์ของ packet ที่เป็น 1
plt.bar(binary, [X[:, FEATURE_NAMES.index(n)].mean() for n in binary])
# เอียงชื่อแกนนอน 60 องศาให้อ่านได้ ใส่ชื่อแกนตั้งและหัวกราฟ
plt.xticks(rotation=60, ha="right"); plt.ylabel("สัดส่วน packet"); plt.title("feature แบบ 0/1")
# จัดระยะ -> เซฟรูป -> โชว์
plt.tight_layout(); save_fig(fig2, "features_binary.png"); plt.show()
# ใส่รูปทั้งสองลงรายงาน
report("การกระจายของ feature", "![](features_hist.png)\n\n![](features_binary.png)")

## 4. เทรน StandardScaler + Isolation Forest
เทรนด้วย traffic ปกติอย่างเดียว (unsupervised) — ไม่ต้องใช้ label

In [ ]:
# * ===================== ขั้น 4: เทรนโมเดล =====================
# * เทรนจาก traffic ปกติอย่างเดียว ไม่ต้องมีเฉลย (unsupervised)

# จดเวลาเริ่ม
t0 = time.time()
# เทรน 2 อย่าง: scaler จำค่าเฉลี่ยกับความกว้างของทุกคอลัมน์ (ปรับหน่วยให้เท่ากัน) แล้ว Isolation Forest ปลูกต้นไม้ 1000 ต้นจากข้อมูลนั้น
scaler, model = train_model(X, N_ESTIMATORS, SEED)
# ลองให้คะแนนข้อมูลเทรนเองดู (คะแนนยิ่งสูงยิ่งแปลก) ไว้เช็กว่าโมเดลดูปกติไหม
s_train = anomaly_scores(model, scaler, X)
# โชว์เวลาที่ใช้ และคะแนนของข้อมูลเทรน: ค่ากลาง, ที่ 99%, สูงสุด
print(f"เทรนเสร็จใน {time.time() - t0:.1f}s | score ข้อมูลเทรน: median={np.median(s_train):.3f} "
      f"p99={np.quantile(s_train, .99):.3f} max={s_train.max():.3f}")
# จดตัวเลขเดียวกันลงรายงาน
report("เทรน", f"ใช้เวลา {time.time() - t0:.1f}s | score ข้อมูลเทรน: median={np.median(s_train):.3f} "
       f"p99={np.quantile(s_train, .99):.3f} max={s_train.max():.3f}")

## 5. ชุดทดสอบที่มี label
- **จำลอง** (`USE_SIM_TEST`): traffic ปกติชุดใหม่ (seed ต่างจากตอนเทรน) + ping + nmap หลายแบบ
- **จาก pcap** ใน `training/pcap/test/`: ติด label ด้วยกฎใน `labels.csv` (packet ที่ไม่ตรงกฎ = ปกติ)

In [ ]:
# * ===================== ขั้น 5: เตรียมข้อสอบพร้อมเฉลย =====================
# * ข้อสอบมี 3 แหล่ง: การโจมตีปลอม, CIC วันศุกร์, traffic บ้านช่วงท้าย
# * เก็บแค่ตัวเลขกับเฉลย ไม่เก็บ packet ดิบไว้ ประหยัดแรม

# อ่านกฎทำเฉลยจาก labels.csv (IP ผู้โจมตี + ช่วงเวลา)
rules = ev.load_label_rules(LABELS_FILE)
# รายการข้อสอบ แต่ละชุดเก็บ (ชื่อชุด, ตารางตัวเลข, เฉลยรายแถว, ยาวกี่ชั่วโมง)
test_parts = []

# ถ้าเปิดใช้ข้อสอบปลอม
if USE_SIM_TEST:
    # สร้าง traffic ปกติปลอม 4,000 ตัว (เลขสุ่มคนละชุดกับตอนเทรน) + ping 4 แบบ + nmap 6 แบบ แล้วแปลงเป็นตัวเลขพร้อมเฉลย
    Xs, _, ns = ev.blocks_to_xy(ev.simulated_test_set(4000, SEED + 4200, WINDOW_SEC), WINDOW_SEC)
    # เก็บเป็นข้อสอบชุดแรก (ข้อมูลปลอมไม่มีความยาวเป็นชั่วโมงจริง เลยใส่ None)
    test_parts.append(("จำลอง (ping + nmap)", Xs, ns, None))

# วนทีละไฟล์ข้อสอบจริง
for csv_path in test_csvs:
    # อ่านไฟล์แล้วเรียงตามเวลา (ไม่ตัด ping เพราะข้อสอบต้องเหมือนของจริง)
    pk = sorted(load_packets_csv(csv_path), key=lambda p: p["ts"])
    # ถ้าไฟล์ว่าง
    if not pk:
        # ข้ามไปไฟล์ถัดไป
        continue
    # ทำเฉลยทีละ packet: ตรงกฎข้อไหนก็ได้ชื่อการโจมตีนั้น ไม่ตรงเลย = "normal"
    names = np.array(ev.label_packets(pk, rules), dtype=object)
    # นับว่ามี packet ที่เป็นการโจมตีกี่ตัว
    n_att = int((names != ev.NORMAL).sum())
    # โชว์ชื่อไฟล์ จำนวน packet และจำนวนที่เป็นการโจมตี
    print(f"{csv_path.name}: {len(pk):,} packets, ติด label โจมตี {n_att:,}")
    # ถ้าไฟล์กฎว่างเปล่า
    if not rules:
        # เตือนว่าทุก packet จะถูกนับเป็นปกติหมด
        print("  ⚠️ labels.csv ยังไม่มีกฎ -> ทุก packet ถูกนับเป็นปกติ")
    # ถ้ามีกฎแต่ไม่มี packet ไหนตรงเลย (และไม่ใช่ไฟล์ soak_ ที่ปกติล้วนอยู่แล้ว)
    elif n_att == 0 and not csv_path.name.startswith("soak_"):
        # เตือนให้เช็ก IP เวลา หรือ timezone ในกฎ
        print("  ⚠️ ไม่มี packet ไหนตรงกฎใน labels.csv -> เช็ก IP / ช่วงเวลา / timezone ในกฎ และ pcap_time_windows ใน config")
    # คิดว่าไฟล์นี้ยาวกี่ชั่วโมง (เวลา packet สุดท้าย - packet แรก) ไว้คิดแจ้งเตือนหลอกต่อชั่วโมง
    hours = (pk[-1]["ts"] - pk[0]["ts"]) / 3600
    # แปลงเป็น 36 ตัวเลขด้วยสูตรเดิม แล้วเก็บคู่กับเฉลย
    test_parts.append((csv_path.name, build_features(pk, WINDOW_SEC), names, hours))
    # ทิ้งรายการ packet ดิบทันที คืนแรม (บางไฟล์มีเป็นล้าน packet)
    del pk
# ถ้าไม่มีข้อสอบสักชุด หยุดเลยพร้อมบอกวิธีแก้
assert test_parts, "ไม่มีชุดทดสอบ: ตั้ง USE_SIM_TEST = True หรือวาง pcap ใน training/pcap/test/"

# ทำตารางสรุปข้อสอบแต่ละชุด: กี่ packet, เป็นการโจมตีกี่ตัว, ยาวกี่ชั่วโมง
test_info = [{"ชุด": n, "packets": len(ns), "ติด label โจมตี": int((ns != ev.NORMAL).sum()),
              "ชั่วโมง": round(h, 2) if h else "-"} for n, _, ns, h in test_parts]
# เอาตัวเลขของข้อสอบทุกชุดมาต่อกันเป็นตารางเดียว (ต่อตามลำดับชุด)
X_test = np.vstack([p[1] for p in test_parts])
# เอาเฉลยของทุกชุดมาต่อกันตามลำดับเดียวกัน
names_test = np.concatenate([p[2] for p in test_parts])
# แปลงเฉลยเป็นเลข: 0 = ปกติ, 1 = โจมตี
y_test = (names_test != ev.NORMAL).astype(int)
# ให้โมเดลให้คะแนนข้อสอบทุกแถว (ใช้ scaler ตัวเดิมจากตอนเทรน ไม่ได้ปรับใหม่)
s_test = anomaly_scores(model, scaler, X_test)
# โชว์ว่าข้อสอบมีทั้งหมดกี่ตัว ปกติกี่ตัว โจมตีกี่ตัว
print(f"ชุดทดสอบ: {len(y_test):,} packets | ปกติ {int((y_test == 0).sum()):,} | โจมตี {int(y_test.sum()):,}")
# จดลงรายงาน
report("ชุดทดสอบ", ev.md_table(test_info) + f"\n\nรวม {len(y_test):,} packets | ปกติ {int((y_test == 0).sum()):,} "
       f"| โจมตี {int(y_test.sum()):,}\n\nกฎใน labels.csv: {len(rules)} ข้อ")

## 6. เลือก threshold
- **recommended** (ค่าที่ใช้จริง): หา score ของ traffic ปกติที่ quantile `fp_quantile` (p_norm) กับ score ของ ping ต่ำสุด (ping_min) แล้ววางเส้นที่ **25% ของช่องว่างนับจากฝั่งปกติ** = p_norm + 0.25 × (ping_min − p_norm) — ไม่ได้ดูชุดทดสอบ (ยุติธรรม) ที่ไม่วางกึ่งกลางเพราะการสแกนบางแบบ (เช่นไล่ host) ได้ score ต่ำกว่า ping
- **best_f1**: ค่าที่ให้ F1 สูงสุดบนชุดทดสอบ — ⚠️ ได้ตัวเลขสวยเกินจริง เพราะเลือกจากชุดที่ใช้วัดผลเอง (แสดงไว้เทียบเท่านั้น)

In [ ]:
# * ===================== ขั้น 6: เลือกเส้นตัด (threshold) =====================
# * โมเดลให้แค่คะแนน เราต้องตัดสินเองว่าคะแนนเท่าไหร่ถึงนับว่าแปลก
# * ค่าที่ใช้จริงเลือกจากข้อมูลชุดแยก (validation) ไม่ได้แอบดูข้อสอบ

# สร้าง traffic ปกติปลอมชุดใหม่ 4,000 ตัวที่โมเดลไม่เคยเห็น แล้วตัด ICMP ออก
# (ICMP ปกติหายาก คะแนนเลยสูงใกล้ ping ถ้าใส่ไว้ เส้นตัดจะโดนดันสูงจนจับ ping ไม่ได้)
val = [p for p in generate(4000, SEED + 1) if p["proto"] != "ICMP"]
# ให้คะแนน traffic ปกติชุดนี้
s_val = anomaly_scores(model, scaler, build_features(val, WINDOW_SEC))
# ให้คะแนน ping ปลอม 4 แบบ เอาเฉพาะ ping ขาที่คนอื่นยิงเข้ามา
s_ping = ping_request_scores(model, scaler, WINDOW_SEC)
# หาเส้นตัด: เอาคะแนนปกติที่ 99.8% กับคะแนน ping ต่ำสุด แล้ววางเส้นที่ 25% ของช่องว่างนับจากฝั่งปกติ
thr_rec, note = recommend_threshold(s_val, s_ping, FP_QUANTILE)
# หาเส้นตัดอีกแบบที่ได้คะแนน F1 สูงสุดในข้อสอบ (แอบดูข้อสอบ ใช้แค่เทียบ ไม่ได้ใช้จริง)
thr_f1 = ev.best_f1_threshold(y_test, s_test)

# เลือกเส้นตัดตามสวิตช์: ถ้าเป็นชื่อก็เลือกตามชื่อ ถ้าเป็นตัวเลขก็ใช้ตัวเลขนั้นเลย แล้วแปลงเป็นทศนิยม
THRESHOLD = float({"recommended": thr_rec, "best_f1": thr_f1}.get(THRESHOLD_MODE, THRESHOLD_MODE))
# โชว์เส้นตัดแบบแนะนำ พร้อมบอกว่าแยก ping ได้ไหม (ควรขึ้น OK)
print(f"recommended = {thr_rec:.4f}  ({note})")
# โชว์เส้นตัดแบบ F1 สูงสุด
print(f"best_f1     = {thr_f1:.4f}")
# โชว์เส้นตัดที่ใช้จริง
print(f"ใช้ THRESHOLD = {THRESHOLD:.4f}  (โหมด {THRESHOLD_MODE})")

# เตรียมเส้นตัดหลายค่าไว้ลองเทียบ: 0.55, 0.57 ... 0.75 บวกสองค่าข้างบน (ตัดซ้ำแล้วเรียง)
grid = sorted(set(np.round(np.arange(0.55, 0.76, 0.02), 3).tolist() + [round(thr_rec, 4), round(thr_f1, 4)]))
# ลองตัดข้อสอบด้วยแต่ละค่า แล้วนับผล (จับถูก/หลอก/หลุด)
sweep = ev.threshold_sweep(y_test, s_test, grid)
# เลือกคอลัมน์ที่จะโชว์ในตาราง
cols = ["threshold", "TP", "FP", "TN", "FN", "Accuracy", "Precision", "Recall (TPR)", "F1", "FPR"]
# ทำเป็นตาราง
sweep_md = ev.md_table([{c: r[c] for c in cols} for r in sweep])
# โชว์ตาราง: เส้นสูงขึ้น = แจ้งเตือนหลอกน้อยลง แต่จับพลาดมากขึ้น
display(Markdown("**ผลที่ threshold ต่างๆ** (เลื่อนสูง = alert หลอกน้อยลง แต่จับพลาดมากขึ้น)\n\n" + sweep_md))
# จดลงรายงาน
report("Threshold", f"- recommended = {thr_rec:.4f} ({note})\n- best_f1 = {thr_f1:.4f}\n"
       f"- **ใช้ {THRESHOLD:.4f}** (โหมด {THRESHOLD_MODE})\n\n" + sweep_md)

## 7. ผลการวัด — Confusion Matrix และ Metrics

In [ ]:
# * ===================== ขั้น 7: ตรวจข้อสอบ =====================

# ตัดสินทุกแถว: คะแนนถึงเส้นตัด = 1 (แปลก) ไม่ถึง = 0 (ปกติ) กฎเดียวกับที่ระบบจริงใช้
y_pred = (s_test >= THRESHOLD).astype(int)
# เทียบคำตอบกับเฉลย นับ TP/FP/TN/FN แล้วคิด Accuracy, Precision, Recall, F1, FPR ฯลฯ
m = ev.binary_metrics(y_test, y_pred)
# คิด ROC-AUC เพิ่ม (วัดจากคะแนนตรงๆ ไม่ขึ้นกับเส้นตัด)
m["ROC-AUC"] = ev.auc(y_test, s_test)

# ข้อความตาราง confusion matrix 2x2 พร้อมคำอธิบาย TP/FN/FP/TN ไว้โชว์
cm_md = f"""
### Confusion Matrix (threshold = {THRESHOLD:.4f})
| | ทำนาย: ปกติ | ทำนาย: โจมตี |
|---|---|---|
| **จริง: ปกติ** | TN = {m['TN']:,} | FP = {m['FP']:,} |
| **จริง: โจมตี** | FN = {m['FN']:,} | TP = {m['TP']:,} |

- **TP** จับการโจมตีได้ถูก · **FN** การโจมตีที่หลุด
- **FP** alert หลอก (ปกติแต่ถูกมองว่าแปลก) · **TN** ปกติและตอบว่าปกติ
"""
# ตารางแยกตามชื่อเฉลย: แถว normal = แจ้งเตือนหลอกกี่ %, แถวการโจมตี = จับได้กี่ %
label_md = ev.md_table(ev.per_label_table(names_test, y_pred))

# แยกผลตามข้อสอบแต่ละชุด
# file_rows = ผลของแต่ละชุด, off = ตำแหน่งเริ่มของชุดถัดไป
file_rows, off = [], 0
# วนทีละชุด (ลำดับเดียวกับตอนต่อเป็น X_test)
for tname, _, ns, hours in test_parts:
    # ตัดคำตอบออกมาเฉพาะช่วงของชุดนี้ แล้วเลื่อนตำแหน่งไปชุดถัดไป
    yp = y_pred[off:off + len(ns)]; off += len(ns)
    # ทำเครื่องหมายว่าแถวไหนของชุดนี้เฉลยว่าปกติ
    norm = ns == ev.NORMAL
    # จดผลของชุดนี้: ปกติกี่ตัว, หลอกกี่ตัว, อัตราหลอก, หลอกต่อชั่วโมง, โจมตีกี่ตัว, จับได้กี่ %
    file_rows.append({"ชุด": tname, "packets ปกติ": int(norm.sum()), "packet หลอก": int(yp[norm].sum()),
                      "อัตรา alert หลอก": float(yp[norm].mean()) if norm.any() else 0.0,
                      "packet หลอก/ชม.": round(float(yp[norm].sum()) / hours, 1) if hours else "-",
                      "packets โจมตี": int((~norm).sum()),
                      "จับได้": float(yp[~norm].mean()) if (~norm).any() else "-"})
# ทำเป็นตาราง
file_md = ev.md_table(file_rows)

# โชว์ตารางแยกตามชุด
display(Markdown("### แยกตามชุดทดสอบ\n" + file_md))
# โชว์ confusion matrix
display(Markdown(cm_md))
# โชว์ตาราง metrics
display(Markdown("### Metrics\n" + ev.metrics_md(m)))
# โชว์ตารางแยกตามชื่อเฉลย
display(Markdown("### แยกตาม label\n(แถว normal = อัตรา alert หลอก, แถวโจมตี = อัตราที่จับได้)\n\n" + label_md))
# จดตารางแยกตามชุดลงรายงาน
report("แยกตามชุดทดสอบ", file_md)
# จดผลรวมทั้งหมดลงรายงาน
report("ผลการวัด", cm_md + "\n### Metrics\n" + ev.metrics_md(m) +
       "\n\n### แยกตาม label\n(แถว normal = อัตรา alert หลอก, แถวโจมตี = อัตราที่จับได้)\n\n" + label_md)

# บันทึกตัวเลขผลสอบเป็นไฟล์ metrics.json (แปลงตัวเลขของ numpy เป็นตัวเลขธรรมดาก่อน ไม่งั้น json เขียนไม่ได้)
(RUN_DIR / "metrics.json").write_text(json.dumps(
    {"threshold": THRESHOLD, "threshold_mode": str(THRESHOLD_MODE),
     **{k: (float(v) if isinstance(v, (float, np.floating)) else v) for k, v in m.items()}},
    ensure_ascii=False, indent=2, default=str), encoding="utf-8")

In [ ]:
# * ===================== ขั้น 7 (ต่อ): เกณฑ์ผ่านตาม README =====================
# * ping ต้องโดนจับครบทุกตัว, nmap ต้องโดนจับภายใน 40 packet แรก, ของจริงในบ้านต้องจับได้ 95% ขึ้นไป

# ให้คะแนนการโจมตีปลอมทั้ง 10 แบบ (ping 4 + nmap 6)
scen = score_scenarios(model, scaler, WINDOW_SEC)
# สร้าง packet ชุดเดิมอีกรอบ (ได้ชุดเดิมเป๊ะ เพราะเลขสุ่มตายตัว) ไว้ดูว่าตัวไหนเป็น ping ขาเข้า
_sc = make_scenarios()
# เก็บคะแนนเฉพาะ ping ขาเข้า (ICMP ชนิด 8) ของแต่ละแบบ
ping_req = {k: s[np.array([p["icmp_type"] == 8 for p in _sc[k]])] for k, s in scen.items() if is_ping_scenario(k)}
# ดูการสแกนแต่ละแบบ: packet แรกที่โดนจับคือตัวที่เท่าไร จับได้กี่ % ผ่านเกณฑ์ไหม
scans = scan_report({k: s for k, s in scen.items() if not is_ping_scenario(k)}, THRESHOLD)
# เก็บอัตราจับได้ของแต่ละชื่อเฉลย ไว้เช็กของจริงในบ้าน
lab = {r["label"]: r["อัตรา"] for r in ev.per_label_table(names_test, y_pred)}

# รายการผลเช็กทั้งหมด
checks = []
# เช็ก ping ทีละแบบ
for k, s in ping_req.items():
    # ผ่านเมื่อ ping ขาเข้าทุกตัวได้คะแนนถึงเส้นตัด
    checks.append({"เกณฑ์": f"ping (request) {k}", "ผล": f"{(s >= THRESHOLD).mean():.0%}", "ผ่าน": bool((s >= THRESHOLD).all())})
# เช็กการสแกนทีละแบบ
for k, r in scans.items():
    # ผ่านเมื่อจับได้ภายใน 40 packet แรก
    checks.append({"เกณฑ์": f"{k} จับได้ภายใน {SCAN_DETECT_WITHIN} packet แรก",
                   "ผล": f"packet ที่ {r['first']} (จับได้ {r['rate']:.0%})" if r["first"] else "จับไม่ได้", "ผ่าน": bool(r["ok"])})
# เช็กของจริงในบ้าน 2 อย่าง (ถ้ามีชื่อนี้ใน labels.csv)
for name in ("Ping-iPhone", "PortScan-Fing"):
    # มีเฉลยชื่อนี้ไหม
    if name in lab:
        # ผ่านเมื่อจับได้ 95% ขึ้นไป
        checks.append({"เกณฑ์": f"ของจริงในบ้าน: {name}", "ผล": f"{lab[name]:.1%}", "ผ่าน": bool(lab[name] >= 0.95)})
# ไฟล์ที่ชื่อขึ้นต้น soak_ โชว์จำนวนแจ้งเตือนหลอกเป็นข้อมูล ไม่นับผ่าน/ไม่ผ่าน
for r in file_rows:
    # เฉพาะชุดที่ชื่อขึ้นต้นด้วย soak_
    if r["ชุด"].startswith("soak_"):
        # จดจำนวนแจ้งเตือนหลอกไว้ ช่องผ่านใส่ "-" (ไม่นับ)
        checks.append({"เกณฑ์": f"alert หลอก {r['ชุด']} (ข้อมูล)", "ผล": f"{r['packet หลอก']} packets ({r['packet หลอก/ชม.']}/ชม.)", "ผ่าน": "-"})

# ผ่านทั้งหมดไหม (ไม่นับแถวที่เป็นข้อมูลเฉยๆ)
PASS = all(c["ผ่าน"] for c in checks if c["ผ่าน"] != "-")
# ทำตารางผลเช็ก ต่อท้ายด้วย ✅ ผ่านครบ หรือ ❌ ไม่ผ่าน
gate_md = ev.md_table(checks) + ("\n\n**✅ ผ่านเกณฑ์ README ครบ**" if PASS else "\n\n**❌ ไม่ผ่านเกณฑ์ README — อย่าเอาโมเดลนี้ไปใช้ (กู้โมเดลเดิมจาก backup)**")
# โชว์ตาราง
display(Markdown("### เกณฑ์ผ่านตาม README\n" + gate_md))
# จดลงรายงาน
report("เกณฑ์ผ่านตาม README", gate_md)

In [ ]:
# * ===================== กราฟผลสอบ =====================

# เตรียมพื้นที่กราฟ 3 ช่องเรียงกัน
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
# ช่อง 1: ตาราง confusion matrix เป็นสี
ev.plot_confusion(m, axes[0], f"Confusion Matrix (thr={THRESHOLD:.3f})")
# ช่อง 2: เส้น ROC (ยิ่งโค้งขึ้นมุมซ้ายบนยิ่งดี)
ev.plot_roc(y_test, s_test, axes[1])
# ช่อง 3: คะแนนของปกติ (สีเทา) เทียบกับการโจมตี (สีแดง) และเส้นตัด
ev.plot_score_hist(y_test, s_test, THRESHOLD, axes[2])
# จัดระยะ -> เซฟรูป -> โชว์
fig.tight_layout(); save_fig(fig, "confusion_roc_hist.png"); plt.show()
# ใส่รูปลงรายงาน
report("กราฟ Confusion Matrix / ROC / score", "![](confusion_roc_hist.png)")

> **อ่านผลอย่างไร:** dataset IDS มี traffic ปกติเยอะกว่าการโจมตีมาก **Accuracy อย่างเดียวหลอกตา**
> ให้ดู **Recall** (จับการโจมตีได้กี่ %), **FPR** (alert หลอกกี่ %) และ **F1** ควบคู่
> ส่วน nmap จะหลุดช่วงแรกๆ (FN) เป็นเรื่องปกติ เพราะต้องรอให้ window 5 วินาทีเห็นพฤติกรรมก่อน

## 8. บันทึกโมเดล
บันทึกลง `system/models/` (iforest.joblib, scaler.joblib, meta.json) → detection-service โหลดเมื่อรีสตาร์ท (`detection.threshold: auto` ใน config จะใช้ threshold นี้)
⚠️ ไฟล์เดิมจะถูกเขียนทับ ถ้าโมเดลเดิมดีอยู่ให้สำรองไว้ก่อน (หรือตั้ง `REQUIRE_PASS = True`)

In [ ]:
# * ===================== ขั้น 8: บันทึกโมเดลให้ระบบจริงใช้ =====================
# ! ไฟล์โมเดลเดิมใน system/models จะถูกเขียนทับ

# ถ้าโมเดลสอบไม่ผ่านเกณฑ์
if not PASS:
    # และตั้งไว้ว่าต้องผ่านก่อนถึงจะบันทึก
    if REQUIRE_PASS:
        # หยุดเลย ไม่บันทึก
        raise RuntimeError("ไม่ผ่านเกณฑ์ README -> ไม่บันทึกโมเดล (ตั้ง REQUIRE_PASS = False ถ้าต้องการบันทึกอยู่ดี)")
    # ไม่ได้บังคับ ก็บันทึกต่อ แต่เตือนไว้ก่อน
    print("⚠️ โมเดลนี้ไม่ผ่านเกณฑ์ README ครบ แต่ยังบันทึกเพราะ REQUIRE_PASS = False")

# ข้อมูลเสริมที่จะเขียนลง meta.json: ใช้เส้นตัดแบบไหน, ผลสอบ (ปัดทศนิยม 5 ตำแหน่ง), ข้อสอบกี่ packet
extra = {
    "threshold_mode": str(THRESHOLD_MODE),
    "test_metrics": {k: (round(v, 5) if isinstance(v, float) else v) for k, v in m.items()},
    "test_packets": int(len(y_test)),
}
# รวมชื่อไฟล์ที่ใช้เทรนเป็นข้อความเดียว
src_names = ", ".join(Path(s).name for s in sources)
# บันทึก 3 ไฟล์ลง system/models: โมเดล (iforest.joblib), scaler (scaler.joblib), meta.json (รายชื่อ feature, เวอร์ชัน, เส้นตัด ฯลฯ)
paths = save_artifacts(model, scaler, THRESHOLD, len(X), src_names, cfg, extra)
# โชว์ว่าแต่ละไฟล์ไปอยู่ที่ไหน
for k, p in paths.items():
    # พิมพ์ชื่อไฟล์กับที่อยู่ทีละบรรทัด
    print(f"{k:12} -> {p}")
# ถ้ามี meta.json จริง
if Path(paths.get("meta_path", "")).exists():
    # ก๊อปเก็บไว้ในโฟลเดอร์ผลรอบนี้ด้วย (เพราะใน system/models จะโดนทับรอบหน้า)
    shutil.copy(paths["meta_path"], RUN_DIR / "meta.json")
# จดที่อยู่ไฟล์และแหล่งข้อมูลลงรายงาน
report("บันทึกโมเดล", "\n".join(f"- {k}: `{p}`" for k, p in paths.items()) + f"\n- แหล่งข้อมูลเทรน: {src_names}")
# โชว์ว่าผลรอบนี้อยู่ที่ไหน
print("ผลการเทรนรอบนี้ (รูป + report.md):", RUN_DIR)
# ถ้าในไฟล์ตั้งค่า เส้นตัดถูกตั้งเป็นตัวเลขตายตัว (ไม่ใช่ auto)
if str(cfg["detection"].get("threshold", "auto")).lower() != "auto":
    # เตือนว่าระบบจริงจะใช้ตัวเลขในไฟล์ตั้งค่า ไม่ใช่ค่าที่เพิ่งบันทึก
    print("⚠️ config detection.threshold เป็นตัวเลข -> detection-service จะใช้ค่าใน config แทนค่าที่บันทึกนี้")
# เตือนว่าต้องปิดแล้วเปิดระบบใหม่ ระบบถึงจะโหลดโมเดลใหม่
print("\nขั้นต่อไป: ปิด run_all.py เดิม (Ctrl+C) แล้วรันใหม่ เพื่อโหลดโมเดลใหม่")